In [1]:
# --- Assignment 4: Random Forest Supervised Learning ---
# Author: Nikhil Sangamkar

# ==============================
# 1. Setup & Imports
# ==============================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn_extra.cluster import KMedoids

# --- directories ---
data_dir = Path("data") / "SRP192714"
results_dir = Path("results")
plots_dir = Path("plots")
for d in (results_dir, plots_dir):
    d.mkdir(parents=True, exist_ok=True)

# ==============================
# 2. Load & Clean Metadata
# ==============================
expr = pd.read_csv(data_dir / "SRP192714.tsv", sep="\t", index_col=0)
meta = pd.read_csv(data_dir / "metadata_SRP192714_merged.tsv", sep="\t", encoding="latin-1")

print("Loaded expression and metadata")
print(expr.shape, meta.shape)

# clean exposure labels
exposure_clean = (
    meta["Exposure"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "denv.prior.inf": "denv",
        "denv prior inf": "denv",
        "denv prior infection": "denv",
        "denv+": "denv",
        "naïve": "naive",
        "naive": "naive",
    })
)
meta["Exposure_Clean"] = exposure_clean
meta["Exposure_Pretty"] = meta["Exposure_Clean"].map({"naive": "Naive", "denv": "DENV"})
meta["Exposure_Binary"] = meta["Exposure_Clean"].map({"naive": 0, "denv": 1})

# filter to only valid samples
valid_exposures = {"naive", "denv"}
kept = meta.loc[meta["Exposure_Clean"].isin(valid_exposures), "refinebio_accession_code"]
expr = expr[kept]
meta = meta.set_index("refinebio_accession_code").loc[kept]

print(f"After filtering: {expr.shape[1]} samples, {expr.shape[0]} genes")

# ==============================
# 3. Preprocessing & Top Genes
# ==============================
# detect if log-transform needed
if expr.max().max() > 100:  # likely raw counts
    expr_proc = np.log1p(expr)
    print("🔹 Applied log1p transform (raw counts detected)")
else:
    expr_proc = expr.copy()
    print("🔹 Data already normalized (no log transform)")

gene_var = expr_proc.var(axis=1, ddof=0).sort_values(ascending=False)
top5000 = gene_var.head(5000).index
expr_top = expr_proc.loc[top5000]

print(f"Selected top 5000 variable genes: {expr_top.shape}")

# ==============================
# 4. PAM-style Clustering (K-Medoids)
# ==============================
X_cluster = expr_top.T
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# use 3 clusters like Assignment 3 (adjust if you used a different k)
k = 3
pam = KMedoids(n_clusters=k, random_state=42, method="pam", metric="euclidean")
clusters = pam.fit_predict(X_scaled)

meta["cluster"] = clusters.astype(str)
meta["cluster"].value_counts()

meta[["Exposure_Pretty", "cluster"]].head()

# ==============================
# 5. Random Forest: Exposure Prediction
# ==============================
X = expr_top.T
y = meta.loc[X.index, "Exposure_Pretty"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

rf = RandomForestClassifier(
    n_estimators=500,
    max_features=50,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# check class order
proba = rf.predict_proba(X_test)
didx = list(rf.classes_).index("DENV")
auc_exposure = roc_auc_score((y_test == "DENV").astype(int), proba[:, didx])
print(f"\nAUC for Exposure classification (Naive vs DENV): {auc_exposure:.3f}")

# plot ROC
fpr, tpr, _ = roc_curve((y_test == "DENV").astype(int), proba[:, didx])
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {auc_exposure:.3f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Random Forest ROC – Exposure2 (Naive vs DENV)")
plt.legend()
plt.tight_layout()
plt.savefig(plots_dir / "rf_exposure_roc.png", dpi=300)
plt.close()

# ==============================
# 6. Random Forest: Cluster Prediction (1-vs-all)
# ==============================
auc_results = []
for c in sorted(meta["cluster"].unique()):
    y_cluster = np.where(meta["cluster"] == c, 1, 0)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_cluster, test_size=0.2, stratify=y_cluster, random_state=42
    )

    rf.fit(X_train, y_train)
    y_prob = rf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    auc_results.append((c, auc))
    print(f"Cluster {c}: AUC = {auc:.3f}")

auc_df = pd.DataFrame(auc_results, columns=["Cluster", "AUC"])
auc_df.to_csv(results_dir / "rf_cluster_auc.tsv", sep="\t", index=False)

# ==============================
# 7. Gene Subset Experiment
# ==============================
gene_counts = [10, 100, 1000, 5000]
subset_results = []

for n in gene_counts:
    topn = gene_var.head(n).index
    X_sub = expr_proc.loc[topn].T

    X_train, X_test, y_train, y_test = train_test_split(
        X_sub, y, test_size=0.2, stratify=y, random_state=42
    )

    rf.fit(X_train, y_train)
    proba = rf.predict_proba(X_test)
    auc_n = roc_auc_score(
        (y_test == "DENV").astype(int),
        proba[:, list(rf.classes_).index("DENV")]
    )
    subset_results.append((n, auc_n))
    print(f"Top {n} genes → AUC = {auc_n:.3f}")

subset_df = pd.DataFrame(subset_results, columns=["NumGenes", "AUC"])
subset_df.to_csv(results_dir / "rf_gene_subset_auc.tsv", sep="\t", index=False)

# plot
plt.figure(figsize=(6, 4))
sns.lineplot(x="NumGenes", y="AUC", data=subset_df, marker="o")
plt.xscale("log")
plt.xlabel("Number of Genes (log scale)")
plt.ylabel("AUC")
plt.title("Effect of Gene Count on Random Forest Performance")
plt.tight_layout()
plt.savefig(plots_dir / "rf_gene_subset_performance.png", dpi=300)
plt.close()

# ==============================
# 8. Final Summary
# ==============================
print("\nRandom Forest analysis complete.")
print("Exposure AUC:", round(auc_exposure, 3))
print("\nCluster prediction AUCs:")
print(auc_df)
print("\nAUC by gene subset:")
print(subset_df)


Loaded expression and metadata
(43363, 1021) (1021, 45)
After filtering: 1021 samples, 43363 genes
🔹 Applied log1p transform (raw counts detected)
Selected top 5000 variable genes: (5000, 1021)

AUC for Exposure classification (Naive vs DENV): 0.948
Cluster 0: AUC = 0.959
Cluster 1: AUC = 0.999
Cluster 2: AUC = 0.990
Top 10 genes → AUC = 0.964
Top 100 genes → AUC = 0.986
Top 1000 genes → AUC = 0.990
Top 5000 genes → AUC = 0.948

Random Forest analysis complete.
Exposure AUC: 0.948

Cluster prediction AUCs:
  Cluster       AUC
0       0  0.959082
1       1  0.998897
2       2  0.990006

AUC by gene subset:
   NumGenes       AUC
0        10  0.964443
1       100  0.985768
2      1000  0.989745
3      5000  0.948294
